# Сбор и подготовка данных по калорийности (БЖУ) для ингредиентов

В качестве источника данных используется открытые данные USDA (U.S. DEPARTMENT OF AGRICULTURE) "Agricultural Research Service":
1) Старница "Download FoodData Central Data" - https://fdc.nal.usda.gov/download-datasets
    - Документация - https://fdc.nal.usda.gov/data-documentation
    - Переход в "Food and Nutrient Database for Dietary Studies (FNDDS)""https://www.ars.usda.gov/northeast-area/beltsville-md-bhnrc/beltsville-human-nutrition-research-center/food-surveys-research-group/docs/fndds-download-databases/
    - загрузка "2021-2023 FNDDS At A Glance - FNDDS Nutrient Values.xlsx" - https://www.ars.usda.gov/ARSUserFiles/80400530/apps/2021-2023%20FNDDS%20At%20A%20Glance%20-%20FNDDS%20Nutrient%20Values.xlsx
    
2) Из документа "FNDDS Nutrient Values.xlsx" сделан рабочий вариант (csv) для возможности обработки - `data/usda/usda_nutrient_values.csv`
    - оставлена только информация по калориям и БЖУ выгружены - `nutr_ds` - 5431 блюд

3) Обработка и сопоставление с заданными в проекте ингредиентами  (`data/ingredients.csv`):
    - 1-ый этап матчинга 
        - поиск включения названия ингредиента (`ingr_ds.ingr`) в описание блюда (`nutr_ds.food_descr`) - если найдено, то заполняется колонка `food_ingr`
        - таким образом первое сопадание записывается напротив блюда, остальные блюда удаляются
        - результат = 3207 записей (содержит в том числе все найденные похожие блюда и дубли)- сохранен в `data/nutrienst/raw/ingr_nutr_mapping_raw.csv`
        - удалось сматчить 122 уникальных ингредиента
        - всего активных (которые есть в проектных блюдах) 199 ингредиентов, значит остается 77 которые не были сопоставлены - `data/nutrients/raw/ingr_nutr_mapping_raw_not_found.csv`
    - 2-ой этап 
        - очистка дублей после автоматического поиска - 3207 записи -> надо свести в 122 уникальных описания нутриента по данному ингредиенту
            - это выполнено вручную путем визуального сканирования и выбора лучшего соответсвия ингредиента и нутриента, чтобы оставить только одну запись как источник калорий и БЖУ по ингредиенту
        - дополнительное сопоставление 77 ингредиентов, которые не были сопоставлены 1-м прогоном
            - это выполнено вручную в том же исходном маппинге с 3207 записями 
                - почти по всем из 77 найдены названия аналоги, или похожие продукты, которые указаны в файле вручную                
                - тех кто не был найден, внесен в маппинг вручную в позиции с близкой калорийностью и близкими по смыслу продуктам с помощью поиска в интернете
    - результат обработки сохранен в `data/nutrients/processed/ingr_nutr_mapping_processed.csv`
        - смапирпованы 197 ингредиента, остались только "ненастоящие" ингредиенты - пустая тарелка ("plate only") и удаленные (deprecated)

    - стоит иметь в виду, что соответсвие не везде полное, поэтому считать что эти данные по калориям на 100% совпадают с заданным датасетом нельзя


In [1]:
# Подготовка датасета `data/usda/usda_nutrient_values.csv`

import pandas as pd
from pathlib import Path

nutr_path = Path('data/usda/usda_nutrient_values.csv')
nutr_ds = pd.read_csv(nutr_path, sep=';')
nutr_ds = nutr_ds.rename(columns={
    'Food code': 'code', 
    'Main food description': 'food_descr', 
    'WWEIA Category number': 'cat_number',
    'WWEIA Category description': 'cat_descr',
    'Energy (kcal)': 'energy_kcal',
    'Protein (g)': 'protein_g',
    'Carbohydrate (g)': 'carb_g',
    'Sugars, total(g)': 'sugars_g',
    'Total Fat (g)': 'fat_g',
    'Water(g)': 'water_g',
    })

nutr_ds

,code,food_descr,cat_number,cat_descr,energy_kcal,protein_g,carb_g,sugars_g,fat_g,water_g
0,11100000,"Milk, NFS",1004,"Milk, reduced fat",52,"3,33","4,83","4,88","2,14","88,92"
1,11111000,"Milk, whole",1002,"Milk, whole",61,"3,27","4,63","4,81","3,2","88,1"
2,11112110,"Milk, reduced fat (2%)",1004,"Milk, reduced fat",50,"3,36","4,9","4,89","1,9","89,1"
3,11112210,"Milk, low fat (1%)",1006,"Milk, lowfat",43,"3,38","5,18","4,96","0,95","89,7"
4,11113000,"Milk, fat free (skim)",1008,"Milk, nonfat",34,"3,43","4,92","5,05","0,08","90,8"
...,...,...,...,...,...,...,...,...,...,...
5426,99997810,Vegetables as ingredient in curry,9999,Not included in a food category,53,"1,81","11,71","3,2","0,2","85,51"
5427,99997815,Vegetables as ingredient in soups,9999,Not included in a food category,50,"1,87","10,56","2,34","0,33","86,48"
5428,99997820,Vegetables as ingredient in stews,9999,Not included in a food category,59,"1,77","12,67","3,16","0,3","84,48"
5429,99998130,Sauce as ingredient in hamburgers,9999,Not included in a food category,272,"1,34","17,14","13,08","22,85","55,97"


In [236]:
# данные по исходным блюдам и ингредиентам проекта по итогам EDA

# блюда
meal_path = Path('data/meal_combined_cleaned.csv')
meal_ds = pd.read_csv(meal_path)

print('Данные по блюдам и ингредиентам')
display(meal_ds)

# ингредиенты
ingr_path = Path('data/meal_ingr.csv')
ingr_ds = pd.read_csv(ingr_path)

print('Список ингредиентов')
display(ingr_ds)

Данные по блюдам и ингредиентам


,dish_id,total_calories,total_mass,split,ing_count,ingr_id,ingr_name
0,dish_1556572657,41.399998,36.0,train,1.0,[12.0],['olives']
1,dish_1556573514,6.440000,23.0,train,1.0,[29.0],['mixed greens']
2,dish_1556575014,71.299995,62.0,train,1.0,[12.0],['olives']
3,dish_1556575083,27.520000,64.0,train,1.0,[16.0],['brussels sprouts']
4,dish_1556575124,4.480000,28.0,train,1.0,[34.0],['celery']
...,...,...,...,...,...,...,...
3225,dish_1568666230,55.800003,268.0,train,2.0,"[27.0, 30.0]","['broccoli', 'lettuce']"
3226,dish_1568666263,93.400002,348.0,train,3.0,"[27.0, 30.0, 95.0]","['broccoli', 'lettuce', 'artichokes']"
3227,dish_1568666303,443.199982,560.0,train,4.0,"[27.0, 28.0, 30.0, 95.0]","['broccoli', 'chicken', 'lettuce', 'artichokes']"
3228,dish_1568666329,743.760010,594.0,train,5.0,"[27.0, 28.0, 30.0, 95.0, 161.0]","['broccoli', 'chicken', 'lettuce', 'artichokes..."


Список ингредиентов


,ingr
0,cottage cheese
1,strawberries
2,garden salad
3,bacon
4,potatoes
...,...
194,tomatillo
195,white beans
196,chia seeds
197,greek yogurt


## Сопоставление названия ингредиента с описанием блюда

In [21]:
# сопоставление названия ингредиента с описанием блюда

def match_ingr(food: str):
    for ingr in ingr_ds['ingr']:
        found = food.find(ingr) != -1
        if found:
            return ingr
    return None

nutr_ds['food_ingr'] = nutr_ds['food_descr'].str.lower().apply(lambda food: match_ingr(food))

# убираем блюда, по которым не нашлось ингредиента
nutr_mapped_ds = nutr_ds[nutr_ds['food_ingr'].notna()]
nutr_mapped_ds = nutr_mapped_ds.sort_values(by=['food_ingr'])

# Сохраним "нутриенты", которые содержат имеющиеся ингредиенты для отбора/маппинга вручную
nutr_mapped_ds.to_csv('data/nutrients/raw/ingr_nutr_mapping_raw.csv', index=False, sep=';')

# результат = 3207 продукта сопоставлены с ингредиентами в 'food_ingr'
nutr_mapped_ds

,code,food_descr,cat_number,cat_descr,energy_kcal,protein_g,carb_g,sugars_g,fat_g,water_g,food_ingr
2106,42101300,"Almonds, flavored",2804,Nuts and seeds,599,"20,07","20,12","4,65","53,27","2,31",almonds
2105,42101130,"Almonds, unsalted",2804,Nuts and seeds,607,"20,33","20,38","4,71","53,96","2,34",almonds
2104,42101120,"Almonds, lightly salted",2804,Nuts and seeds,604,"20,23","20,28","4,69","53,7","2,33",almonds
2103,42101110,"Almonds, salted",2804,Nuts and seeds,601,"20,13","20,18","4,67","53,43","2,32",almonds
2102,42101000,"Almonds, unroasted",2804,Nuts and seeds,626,"21,45","20,03","4,35","51,09","4,26",almonds
...,...,...,...,...,...,...,...,...,...,...,...
46,11434000,"Yogurt, Greek, whole milk, fruit",1822,"Yogurt, Greek",114,"7,85","11,8","8,23","3,91","75,72",yogurt
2700,53710502,Cereal or granola bar (Kellogg's Nutri-Grain Y...,5402,Cereal bars,365,"4,22","67,61","28,98","8,67","16,7",yogurt
4720,75535000,"Zucchini, pickled",8408,"Olives, pickles, pickled vegetables",35,1,"7,44","6,94","0,28","88,11",zucchini
2460,52306550,"Muffin, zucchini",4402,"Biscuits, muffins, quick breads",306,"5,05","42,64","22,72","13,14","37,35",zucchini


In [22]:
# сколько ингредиентов удалось сопоставить с блюдами ("нутриентами")
nutr_ingr_count = nutr_mapped_ds.groupby('food_ingr', dropna=False).count().sort_values(by='code')

# всего 122 найденных соответсвий
# значит 199 - 122 = 77 ингредиента, по которым не найдена информация .. это почти 40%, для ручного маппинга

nutr_ingr_count

,code,food_descr,cat_number,cat_descr,energy_kcal,protein_g,carb_g,sugars_g,fat_g,water_g
food_ingr,,,,,,,,,,
artichokes,1,1,1,1,1,1,1,1,1,1
basil,1,1,1,1,1,1,1,1,1,1
caesar salad,1,1,1,1,1,1,1,1,1,1
buttermilk,1,1,1,1,1,1,1,1,1,1
chia seeds,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...
fish,149,149,149,149,149,149,149,149,149,149
beef,161,161,161,161,161,161,161,161,161,161
bread,220,220,220,220,220,220,220,220,220,220


## Инегредиенты без найденного соответсвия

In [23]:
# посмотрим по каким инегредиентам не найдено соответсвие (77 шт)
 
nutr_not_mapped_ds = ingr_ds[~ingr_ds['ingr'].isin(nutr_mapped_ds['food_ingr'])]

# сохраняем в файл для ручной обработки
nutr_not_mapped_ds.to_csv('data/nutrients/raw/ingr_nutr_mapping_raw_not_found.csv', index=False, sep=';')

nutr_not_mapped_ds


,ingr
7,scrambled eggs
8,wild rice
14,hash browns
24,mixed greens
26,cucumbers
...,...
190,orange with peel
192,nopales
194,tomatillo
197,greek yogurt


## Результат ручной обработки

Все, что не удалось смаппировать автоматически домапировно по похожим блюдам (+ запросы в Google) вручную - `data/nutrients/ingr_nutr_mapping_processed.csv`

In [32]:
# блюда
ingr_mapped_path = Path('data/nutrients/processed/ingr_nutr_mapping_processed.csv')
ingr_mapped_full_ds = pd.read_csv(ingr_mapped_path, sep=';')

display(ingr_mapped_full_ds)
display(ingr_mapped_full_ds['not_match'].value_counts())

,code,cat_number,cat_descr,food_descr,food_ingr,not_match,energy_kcal,protein_g,carb_g,sugars_g,fat_g,water_g
0,91701010,5702.0,Candy containing chocolate,"Almonds, chocolate covered candy",almonds,1,560,"12,97","44,04","32,84","38,81","1,86"
1,42101300,2804.0,Nuts and seeds,"Almonds, flavored",almonds,1,599,"20,07","20,12","4,65","53,27","2,31"
2,42101350,2804.0,Nuts and seeds,"Almonds, honey roasted",almonds,1,579,"14,11","35,84","25,01","45,34","1,63"
3,42101120,2804.0,Nuts and seeds,"Almonds, lightly salted",almonds,1,604,"20,23","20,28","4,69","53,7","2,33"
4,42100100,2804.0,Nuts and seeds,"Almonds, NFS",almonds,0,598,"20,96","21,01","4,86","52,54","2,41"
...,...,...,...,...,...,...,...,...,...,...,...,...
3203,11411100,1820.0,"Yogurt, regular","Yogurt, whole milk, plain",yogurt,1,78,"3,82","5,57","4,09","4,48","85,27"
3204,52306550,4402.0,"Biscuits, muffins, quick breads","Muffin, zucchini",zucchini,1,306,"5,05","42,64","22,72","13,14","37,35"
3205,53124110,5502.0,Cakes and pies,"Cake or cupcake, zucchini",zucchini,1,371,"2,52","51,08","38,27","17,52","27,4"
3206,75535000,8408.0,"Olives, pickles, pickled vegetables","Zucchini, pickled",zucchini,0,35,1,"7,44","6,94","0,28","88,11"


not_match
1    3011
0     197
Name: count, dtype: int64

In [ ]:
# убираем лишние продукты справочника USAD, которые не были смапированы, оставляем только сматченные продукты - 197 шт

ingr_mapped_ds = ingr_mapped_full_ds[ingr_mapped_full_ds['not_match'] == 0]
ingr_mapped_ds

,code,cat_number,cat_descr,food_descr,food_ingr,not_match,energy_kcal,protein_g,carb_g,sugars_g,fat_g,water_g
4,42100100,2804.0,Nuts and seeds,"Almonds, NFS",almonds,0,598,"20,96","21,01","4,86","52,54","2,41"
14,63101000,6002.0,Apples,"Apple, raw",apple,0,61,"0,17","14,8","12,08","0,15","84,62"
32,75400500,3104.0,Vegetable dishes,"Artichokes, stuffed",artichokes,0,160,"5,33","22,28","2,13","5,94","64,76"
36,75202013,6420.0,Other vegetables and combinations,"Asparagus, canned, cooked, no added fat",chayote squash,0,19,"2,14","2,46","1,06","0,65","93,98"
40,75202011,6420.0,Other vegetables and combinations,"Asparagus, fresh, cooked, no added fat",bok choy,0,21,"2,36","4,16","2,02","0,13","92,41"
...,...,...,...,...,...,...,...,...,...,...,...,...
3168,11460000,5802.0,Ice cream and frozen dairy desserts,"Frozen yogurt, vanilla",frozen yogurt,0,127,3,"21,6","19,92","3,6","71,2"
3185,11411400,1822.0,"Yogurt, Greek","Yogurt, Greek, whole milk, plain",greek yogurt,0,94,"8,78","4,75","3,25","4,39","81,32"
3193,11400000,1820.0,"Yogurt, regular","Yogurt, NFS",yogurt,0,94,"6,67","10,22","8,05","2,87","79,37"
3206,75535000,8408.0,"Olives, pickles, pickled vegetables","Zucchini, pickled",zucchini,0,35,1,"7,44","6,94","0,28","88,11"


In [ ]:
# посмотрик какие ингредиенты остались из исходных 199 несмапленных

ingr_ds[~ingr_ds['ingr'].isin(ingr_mapped_ds['food_ingr'])]

# остались только "ненастоящие" ингредиенты - пустая тарелка и удаленные (deprecated)

,ingr
140,plate only
149,deprecated


# Анализ базовых ингредиентов (Весь раздел перенести в EDA)

Проверим сколько есть блюд, с одним ингредиентом - это будут "базовые" ингредиенты

In [ ]:
# выбираем блюда с одним ингредиентом и убираем лишние символы
meal_one_ds = meal_ds[meal_ds['ing_count'] == 1].copy()
meal_one_ds['ingr_name'] = meal_one_ds['ingr_name'].str.replace(r"""['\]\[]""", '', regex=True)

# убираем ненастоящие ингредиенты
meal_one_ds = meal_one_ds[~meal_one_ds['ingr_name'].isin(['plate only', 'deprecated'])]

# рассчитываем реальную калорийность на 100г
meal_one_ds['cal_100g_real'] = meal_one_ds['total_calories'] / (meal_one_ds['total_mass']/100)

meal_one_ds = meal_one_ds.sort_values('ingr_name')
meal_one_ds

# Видим, что у нас несколько образцов одного ингредиента
# 892 продукта - а уникальных ингредиентов 197, значит есть несколько блюд с одним и тем же ингредиентом

,dish_id,total_calories,total_mass,split,ing_count,ingr_id,ingr_name,cal_100g_real
134,dish_1558375667,687.939026,119.0,test,1.0,[77.0],almonds,578.100022
95,dish_1558114523,104.057999,18.0,train,1.0,[77.0],almonds,578.099994
165,dish_1558381032,115.620003,20.0,train,1.0,[77.0],almonds,578.100015
473,dish_1558722125,497.166016,86.0,test,1.0,[77.0],almonds,578.100019
471,dish_1558722068,179.210999,31.0,train,1.0,[77.0],almonds,578.099997
...,...,...,...,...,...,...,...,...
67,dish_1558031133,136.510010,110.0,test,1.0,[192.0],yam,124.100009
935,dish_1560527745,87.761002,149.0,train,1.0,[262.0],yogurt,58.900001
964,dish_1560788349,78.925995,134.0,train,1.0,[262.0],yogurt,58.899996
963,dish_1560788338,71.268997,121.0,train,1.0,[262.0],yogurt,58.899998


In [205]:
# сколько всего базовых ингредиентов и как образцы разделены по сплитам

print(f'Блюда с единственным (базовым) ингредиентом всего = {meal_one_ds["ingr_name"].nunique()}')
# 89 - то есть чуть меньше половины из 197 задана точно на блюдах

ingr_train = meal_one_ds[meal_one_ds['split'] == 'train']
print(f'Базовые ингредиенты в train = {ingr_train["ingr_name"].nunique()}')
# 84 - заданы на train

ingr_test = meal_one_ds[meal_one_ds['split'] == 'test']
print(f'Базовые ингредиенты в test = {ingr_test["ingr_name"].nunique()}')
# 5 - заданы на test

test_only_single = set(ingr_test["ingr_name"]) - set(ingr_train["ingr_name"])
print(f'Базовые ингредиенты только в test = {len(test_only_single)}, {test_only_single}')

Блюда с единственным (базовым) ингредиентом всего = 89
Базовые ингредиенты в train = 84
Базовые ингредиенты в test = 50
Базовые ингредиенты только в test = 5, {'raspberries', 'tuna salad', 'bread', 'wild rice', 'blackberries'}


## Анализ - сколько блюд не являются составными из базовых
- если все сложные блюда составлены из ингредиентов, каждый из которых является базовым
    - то модель будет иметь возможность "вычислять" их сумму, опираясь информацию из базовых блюд
- если же составные блюда содержат ингредиенты, которые не были представлены по отдельности (базовыми)
    - то модели будет сложно предсказывать две неизвестные величины 
        - их массу по отдельности и калорийность каждого продукта
    - для таких продуктов как раз будет хорошая подсказка наличие данных внешних справочников!
        - опираясь на которые модель сможет доучить (скорректировать) их по предоставленным изображениям

In [ ]:
# meal_cleaned.csv - плоский список блюд (см. этап EDA)
meal_flat_ds = pd.read_csv(Path('data/meal_cleaned.csv'))

meal_train = meal_flat_ds[meal_flat_ds['split'] == 'train']
meal_test = meal_flat_ds[meal_flat_ds['split'] == 'test']

print('Сложные блюда без базовых на train')
print(' - Количество блюд:', meal_train[~meal_train['ingr'].isin(meal_one_ds['ingr_name'])]['dish_id'].nunique())
print(' - Количество ингредиентов:', meal_train[~meal_train['ingr'].isin(meal_one_ds['ingr_name'])]['ingr'].nunique())

print('Сложные блюда без базовых на test')
print(' - Количество блюд: ', meal_test[~meal_test['ingr'].isin(meal_one_ds['ingr_name'])]['dish_id'].nunique())
print(' - Количество ингредиентов:', meal_test[~meal_test['ingr'].isin(meal_one_ds['ingr_name'])]['ingr'].nunique())

# ---
# почти половина таких сложных блюд в каждом сплите и 109 сложных ингредиентов vs 84 базовых ингредиента

Сложные блюда без базовых на train
 - Количество блюд: 1528
 - Количество ингредиентов: 109
Сложные блюда без базовых на test
 - Количество блюд:  251
 - Количество ингредиентов: 83


Вывод - почти половина (1528 train / 251 test) сложных блюд в каждом сплите и (109 train / 83 test ) сложных ингредиентов vs ( 84 train / 50 test) базовых ингредиента.
Это подтверждает, что данные внешних справочников должны быть полезными для 109 продуктов

# Проверка - есть ли ингредиенты, которые присутсвуют только на test (перенести в EDA)
- это проблема для обучения

In [177]:
# проверка - есть ли ингредиенты, которые присутсвуют только на test

# meal_cleaned.csv - плоский список блюд (см. этап EDA)
meal_flat_ds = pd.read_csv(Path('data/meal_cleaned.csv'))
display(meal_flat_ds)

# проверка ингредиентов только на train

meal_train = meal_flat_ds[meal_flat_ds['split'] == 'train']
meal_test = meal_flat_ds[meal_flat_ds['split'] == 'test']

meal_train_only = meal_train[~meal_train['ingr'].isin(meal_test['ingr'])]['ingr'].unique()
print(f'Уникальные ингредиенты только для train: {len(meal_train_only)}, {meal_train_only}\n')
# 41 только на train

# проверка ингредиентов только на test

meal_train = meal_flat_ds[meal_flat_ds['split'] == 'train']
meal_test = meal_flat_ds[meal_flat_ds['split'] == 'test']

meal_test_only = meal_test[~meal_test['ingr'].isin(meal_train['ingr'])]['ingr'].unique()
print(f'Уникальные ингредиенты только для test: {len(meal_test_only)}, {meal_test_only}')
# 2 только на train

,dish_id,total_calories,total_mass,ingredients,split,ing_count,ingr
0,dish_1563468327,351.886047,483.0,1.0,train,7.0,cottage cheese
1,dish_1563468299,210.350006,385.0,1.0,train,4.0,cottage cheese
2,dish_1561741063,132.300003,135.0,1.0,test,1.0,cottage cheese
3,dish_1561393619,234.040009,284.0,1.0,train,2.0,cottage cheese
4,dish_1561393550,172.480011,176.0,1.0,train,1.0,cottage cheese
...,...,...,...,...,...,...,...
23782,dish_1565020895,348.126129,416.0,545.0,test,8.0,greek yogurt
23783,dish_1564588889,234.494431,346.0,545.0,test,5.0,greek yogurt
23784,dish_1562691705,412.676880,554.0,545.0,train,10.0,greek yogurt
23785,dish_1562691737,485.676880,700.0,545.0,train,11.0,greek yogurt


Уникальные ингредиенты только для train: 41, ['pasta salad' 'blue cheese' 'tortilla' 'syrup' 'raisins' 'succotash'
 'brown sugar' 'parsnips' 'grapefruit juice' 'tempeh' 'rice noodles' 'cod'
 'buttermilk' 'ice cream cones' 'vegetable oil' 'mozzarella cheese'
 'wheat bread' 'yogurt' 'sandwiches' 'toast' 'pepperoni' 'flour' 'waffles'
 'turnips' 'baby carrots' 'mangos' 'sunflower seeds' 'chilaquiles'
 'fruit salad' 'tuna' 'greek salad' 'cereal' 'chicken salad' 'kimchi'
 'banana with peel' 'orange with peel' 'jicama' 'nopales' 'tomatillo'
 'white beans' 'balsamic vinegar']

Уникальные ингредиенты только для test: 2, ['bread' 'tuna salad']


## Проблема - ингредиенты только на test

Есть два ингредиента, которые уникальны для test-сплитам - ['bread' 'tuna salad'].
- Это будет проблема для модели, так как она не могла их выучить

In [ ]:
# посмотрим, сколько таких блюд

meal_test[meal_test['ingr'].isin(meal_test_only)]

# имеем 6 разных блюд в test-сплите, которые содержат неизвестный на train продукт 
# 6 / 507 ~ 1.2% от всех тестовых блюд - целесообразно удалить такие блюда, иначе ошибка неизбежна

,dish_id,total_calories,total_mass,ingredients,split,ing_count,ingr
1865,dish_1562085672,122.084000,46.0,19.0,test,1.0,bread
10684,dish_1568401261,530.393372,443.0,141.0,test,6.0,tuna salad
10685,dish_1568401302,716.479370,561.0,141.0,test,7.0,tuna salad
10686,dish_1568401199,459.643372,343.0,141.0,test,4.0,tuna salad
10687,dish_1568401167,426.360016,228.0,141.0,test,1.0,tuna salad
10688,dish_1568401233,490.393372,418.0,141.0,test,5.0,tuna salad


# TODO - перенести в основной ноутбук анализ и фильтрацию

# Грубая проверка соответсвия калорийности справочника

- Проверим калорийность на 100г для базовых блюд с одним ингредиентом как бенчмарк общей точности внешнего справочника
- Для небазовых блюд такую проверку сделать не получится, т.к. их масса (распределение в блюде) численно не определена

## Сверяем калорийность по справочнику с калорийностью датасета

In [223]:
# отберем одну строчку на ингредиент
meal_one_product_ds = meal_one_ds.groupby('ingr_name').agg('first')

# мержим с маппированными ингредиентами-нутриентами
meal_one_merge_ds = (meal_one_product_ds[['dish_id', 'total_calories', 'total_mass', 'split', 'cal_100g_real']]
    .merge(
        ingr_mapped_ds.set_index('food_ingr')[['food_descr', 'energy_kcal']], 
        left_index=True, 
        right_index=True)
)

meal_one_merge_ds

,dish_id,total_calories,total_mass,split,cal_100g_real,food_descr,energy_kcal
almonds,dish_1558375667,687.939026,119.0,test,578.100022,"Almonds, NFS",598
apple,dish_1558550079,73.839996,142.0,train,51.999997,"Apple, raw",61
asparagus,dish_1560370114,7.956000,39.0,train,20.400000,"Asparagus, raw",20
avocado,dish_1558468401,75.200005,47.0,train,160.000011,"Avocado, raw",160
bacon,dish_1558030910,81.149994,15.0,train,540.999960,"Beef, bacon, cooked",449
...,...,...,...,...,...,...,...
wheat berry,dish_1560976044,124.670006,137.0,train,91.000004,"Cereal, shredded wheat, plain",358
white rice,dish_1558549605,97.500000,75.0,test,130.000000,"Rice, white, cooked, made with butter",147
wild rice,dish_1563566909,222.530014,187.0,test,119.000007,"Rice, brown, cooked, made with butter",137
yam,dish_1558110436,85.629005,69.0,test,124.100007,"Sweet potato, boiled, NS as to fat",115


In [ ]:
# считаем разницу

import numpy as np

meal_one_merge_ds['cal_100g_delta_abs'] = np.abs(meal_one_merge_ds['cal_100g_real'] - meal_one_merge_ds['energy_kcal'])
meal_one_merge_ds['cal_100g_delta_percent'] = np.round(meal_one_merge_ds['cal_100g_delta_abs'] / meal_one_merge_ds['cal_100g_real'], 2) * 100

print(f'Средняя ошибка калорийности на весь датасет: {meal_one_merge_ds["cal_100g_delta_abs"].mean()}')

meal_one_merge_ds.sort_values(by=['cal_100g_delta_abs'], ascending=False)

Средняя ошибка калорийности на весь датасет: 27.344943764050527


,dish_id,total_calories,total_mass,split,cal_100g_real,food_descr,energy_kcal,cal_100g_delta_abs,cal_100g_delta_percent
wheat berry,dish_1560976044,124.670006,137.0,train,91.000004,"Cereal, shredded wheat, plain",358,266.999996,293.0
ice cream cones,dish_1560370804,25.134001,6.0,train,418.900017,"Ice cream cone, NFS",229,189.900017,45.0
fish,dish_1557861605,35.099998,39.0,train,89.999995,"Fish, salmon, NFS",274,184.000005,204.0
tuna salad,dish_1568401167,426.360016,228.0,test,187.000007,"Tuna salad, made with light Italian dressing",76,111.000007,59.0
hash browns,dish_1559668238,290.139984,89.0,train,325.999982,"Potato, hash brown, from fresh, with cheese",223,102.999982,32.0
...,...,...,...,...,...,...,...,...,...
chia seeds,dish_1562173111,792.179993,163.0,train,485.999996,Chia seeds,486,0.000004,0.0
watermelon,dish_1559061675,24.900002,83.0,train,30.000002,"Watermelon, raw",30,0.000002,0.0
brussels sprouts,dish_1558724574,50.740002,118.0,train,43.000002,"Brussels sprouts, raw",43,0.000002,0.0
cauliflower,dish_1561579505,31.500000,126.0,train,25.000000,"Cauliflower, raw",25,0.000000,0.0


Вывод:
- средняя ошибка - 27 кал - получается довольно много
- топ-3
    - "wheat berry" (зерна пшеницы) - в датасете - 91 vs 358 (340 в интернете) - разница на 266 кал 
    - "ice cream cones" (вафельный рожок для мороженого) - в датасете 418 vs 229 (120-300+) - разница 189 кал 
    - "fish" (какая-то рыба без уточнения) - в датасете 90 vs 274 (если это лосось) - разница 184 кал

Таким образом датасет может содержать явные ошибки/противоречия.
Поэтому в лоб использовать справочные данные нельзя, но как подсказка, особенно для всякой зелени будет полезно.

## Финальные данные по калорийности

In [241]:
# сохраним итоговый маппинг в файл для конструирования признаков в модели
ingr_mapped_path = Path('data/nutrients/processed/ingr_nutr_mapping_result.csv')

ingr_mapped_ds.to_csv(ingr_mapped_path, index=False)
ingr_mapped_ds

,code,cat_number,cat_descr,food_descr,food_ingr,not_match,energy_kcal,protein_g,carb_g,sugars_g,fat_g,water_g
4,42100100,2804.0,Nuts and seeds,"Almonds, NFS",almonds,0,598,"20,96","21,01","4,86","52,54","2,41"
14,63101000,6002.0,Apples,"Apple, raw",apple,0,61,"0,17","14,8","12,08","0,15","84,62"
32,75400500,3104.0,Vegetable dishes,"Artichokes, stuffed",artichokes,0,160,"5,33","22,28","2,13","5,94","64,76"
36,75202013,6420.0,Other vegetables and combinations,"Asparagus, canned, cooked, no added fat",chayote squash,0,19,"2,14","2,46","1,06","0,65","93,98"
40,75202011,6420.0,Other vegetables and combinations,"Asparagus, fresh, cooked, no added fat",bok choy,0,21,"2,36","4,16","2,02","0,13","92,41"
...,...,...,...,...,...,...,...,...,...,...,...,...
3168,11460000,5802.0,Ice cream and frozen dairy desserts,"Frozen yogurt, vanilla",frozen yogurt,0,127,3,"21,6","19,92","3,6","71,2"
3185,11411400,1822.0,"Yogurt, Greek","Yogurt, Greek, whole milk, plain",greek yogurt,0,94,"8,78","4,75","3,25","4,39","81,32"
3193,11400000,1820.0,"Yogurt, regular","Yogurt, NFS",yogurt,0,94,"6,67","10,22","8,05","2,87","79,37"
3206,75535000,8408.0,"Olives, pickles, pickled vegetables","Zucchini, pickled",zucchini,0,35,1,"7,44","6,94","0,28","88,11"
